# Praktikum Bab 14: Mini Project End-to-End: Komparasi Model, Tuning, dan Deployment
### Laboratorium Machine Learning - Universitas Indraprasta PGRI (Unindra)
**Mata Kuliah:** Machine Learning (3 SKS) | **Penyusun Modul:** Nurfidah Dwitiyanti, M.Si.  
**Mahasiswa:** Anggit Arisa Siwi Pratama | **NPM:** `202343570045` | **Kelas:** XC Ekstensi

---

## Capaian Pembelajaran (CPMK dan Sub-CPMK):
1. **CRISP-DM Utuh**: Mahasiswa mampu memetakan dan menyelesaikan proyek Machine Learning end-to-end mengikuti 6 fase CRISP-DM (*Business Understanding*, *Data Understanding*, *Data Preparation*, *Modeling*, *Evaluation*, *Deployment*).
2. **Komparasi Model Standar Industri**: Melatih dan membandingkan minimal 3 algoritma berbeda (Random Forest, Gradient Boosting, SVM) secara adil menggunakan *Stratified 5-Fold Cross-Validation*.
3. **Hyperparameter Tuning Sistematis**: Mengoptimalkan hyperparameter menggunakan `RandomizedSearchCV` untuk eksplorasi efisien ruang parameter besar.
4. **Multi-Criteria Decision Evaluation**: Memilih model terbaik berdasarkan kombinasi metrik objektif (F1-Score, Recall, PR-AUC) dan kepraktisan operasional (kemudahan interpretasi & latensi inferensi).
5. **Model Serialization & Web Deployment**: Membungkus pipeline lengkap ke dalam file biner (`joblib`) dan membangun aplikasi web interaktif menggunakan **Streamlit** (`app.py`).

## 14.1 & 14.10: Persiapan Pustaka & Konfigurasi Lingkungan
Memuat pustaka standar ekosistem Sains Data Python: `joblib`, `pandas`, `numpy`, `matplotlib`, serta modul-modul evaluasi dan algoritma Scikit-Learn.

In [ ]:
# Praktikum 14.1: Persiapan Library
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report
)

print("Pustaka Sains Data & Machine Learning berhasil dimuat.")

## 14.2 & 14.8: Business Understanding & Pemuatan Dataset
**Studi Kasus**: Prediksi Risiko Gagal Bayar Pinjaman (*Loan Default Prediction*).
Bank ingin menekan potensi kerugian finansial akibat kredit macet dengan memprediksi probabilitas gagal bayar nasabah sebelum pengajuan pinjaman disetujui.
- **Fitur Input**:
  1. `pendapatan_bulanan`: Pendapatan bersih peminjam per bulan (Rp).
  2. `usia`: Usia peminjam (tahun).
  3. `lama_kerja`: Lama masa kerja atau kestabilan usaha (tahun).
  4. `jumlah_pinjaman`: Plafon pinjaman yang diajukan (Rp).
  5. `rasio_cicilan`: Rasio cicilan per bulan terhadap pendapatan (Debt Service Ratio).
  6. `skor_kredit`: Skor riwayat kredit historis nasabah (350 - 850).
- **Target**: `status_default` (0: Lancar, 1: Gagal Bayar / Macet).

In [ ]:
# Praktikum 14.2: Import Dataset & Preprocessing
np.random.seed(42)
n_samples = 1200

pendapatan = np.random.lognormal(mean=16.0, sigma=0.5, size=n_samples)
pendapatan = np.clip(np.round(pendapatan, -5), 3_500_000, 45_000_000)

usia = np.random.randint(21, 62, size=n_samples)
lama_kerja = np.clip((usia - 21) - np.random.randint(0, 10, size=n_samples), 0, 35)

jumlah_pinjaman = np.random.choice([15_000_000, 30_000_000, 50_000_000, 100_000_000, 150_000_000], size=n_samples, p=[0.3, 0.3, 0.2, 0.15, 0.05])
cicilan_bulanan = jumlah_pinjaman / np.random.choice([12, 24, 36, 48], size=n_samples)
rasio_cicilan_pendapatan = cicilan_bulanan / pendapatan

skor_kredit = np.clip(np.round(np.random.normal(loc=650, scale=80, size=n_samples)), 350, 850)

# Formulasi log-odds probabilitas default
logit = (
    - 2.8
    + 3.5 * (rasio_cicilan_pendapatan > 0.40)
    + 1.8 * (rasio_cicilan_pendapatan > 0.30)
    - 0.008 * (skor_kredit - 600)
    - 0.08 * lama_kerja
    + 0.6 * (usia < 25)
    + np.random.normal(0, 0.4, size=n_samples)
)
prob_default = 1 / (1 + np.exp(-logit))
gagal_bayar = (np.random.uniform(0, 1, size=n_samples) < prob_default).astype(int)

df = pd.DataFrame({
    "pendapatan_bulanan": pendapatan,
    "usia": usia,
    "lama_kerja": lama_kerja,
    "jumlah_pinjaman": jumlah_pinjaman,
    "rasio_cicilan": np.round(rasio_cicilan_pendapatan, 3),
    "skor_kredit": skor_kredit,
    "status_default": gagal_bayar
})

print(f"Dimensi Dataset Kredit: {df.shape}")
print(df["status_default"].value_counts(normalize=True).rename("Proporsi Target"))
df.head()

## 14.2 & 14.11: Partisi Data & Pelatihan Komparatif 3 Model
Data dibagi menjadi **Train Set (80%)** dan **Test Set (20%)** menggunakan *Stratified Sampling* untuk menjaga rasio kelas minoritas (gagal bayar) tetap konsisten.
Kemudian dikonfigurasikan ruang parameter untuk **Random Forest**, **Gradient Boosting**, dan **SVM (Pipeline)**.

In [ ]:
# Preprocessing & Split Train-Test
X = df.drop(columns=["status_default"])
y = df["status_default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Konfigurasi 3 Model dan Ruang Hyperparameter
model_configs = {
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [50, 100, 200],
            "max_depth": [4, 6, 10, None],
            "min_samples_leaf": [1, 2, 4],
            "class_weight": ["balanced", None]
        },
        3 # Nilai Interpretability (1-5)
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.03, 0.08, 0.15],
            "max_depth": [3, 4, 5],
            "subsample": [0.8, 1.0]
        },
        2 # Nilai Interpretability (1-5)
    ),
    "SVM (RBF Kernel)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(probability=True, random_state=42))
        ]),
        {
            "svm__C": [0.5, 1.0, 5.0, 10.0],
            "svm__gamma": ["scale", "auto", 0.05, 0.1],
            "svm__class_weight": ["balanced", None]
        },
        2 # Nilai Interpretability (1-5)
    )
}

print(f"Data latih: {X_train.shape[0]} baris | Data uji: {X_test.shape[0]} baris")

## 14.3 & 14.11: Tuning Sistematis dengan `RandomizedSearchCV`
Melakukan tuning hyperparameter menggunakan `RandomizedSearchCV` dengan skema `StratifiedKFold(n_splits=5)`. Metrik optimasi yang digunakan adalah **F1-Score** karena data target bersifat timpang (*imbalanced*).

In [ ]:
# Eksekusi Komparasi 3 Model & Tuning
hasil_terbaik = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nama, (model, params, interp) in model_configs.items():
    search = RandomizedSearchCV(
        model,
        param_distributions=params,
        n_iter=6,
        cv=skf,
        scoring="f1",
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    hasil_terbaik[nama] = {
        "estimator": search.best_estimator_,
        "best_score_cv": search.best_score_,
        "best_params": search.best_params_,
        "interpretability": interp
    }
    print(f"[{nama}] F1-Score CV Terbaik = {search.best_score_:.4f}")
    print(f"  Hyperparameter: {search.best_params_}")

## 14.9 & 14.12: Evaluasi Akhir pada Data Uji & Skor Multi-Kriteria
Menghitung metrik F1-score pada data uji (*test set*), Recall, ROC-AUC, serta formula pembobotan multi-kriteria:
$$\text{Skor Gabungan} = 0.7 \times F_1 + 0.3 \times \left(\frac{\text{Interpretability}}{5}\right)$$

In [ ]:
# Praktikum 14.12: Evaluasi pada Data Uji
ringkasan_evaluasi = []

for nama, data_m in hasil_terbaik.items():
    model = data_m["estimator"]
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    f1_test = f1_score(y_test, y_pred)
    rec_test = recall_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc_val = auc(fpr, tpr)
    
    # Formula Skor Terboboti Modul 14.9
    skor_gabungan = 0.7 * f1_test + 0.3 * (data_m["interpretability"] / 5.0)
    data_m["skor_gabungan"] = skor_gabungan
    data_m["f1_test"] = f1_test
    
    ringkasan_evaluasi.append({
        "Model": nama,
        "CV F1": f"{data_m['best_score_cv']*100:.2f}%",
        "Test F1": f"{f1_test*100:.2f}%",
        "Recall": f"{rec_test*100:.2f}%",
        "ROC-AUC": f"{roc_auc_val:.4f}",
        "Interpretability": f"{data_m['interpretability']}/5",
        "Skor Gabungan": f"{skor_gabungan:.4f}"
    })

df_hasil = pd.DataFrame(ringkasan_evaluasi)
print("=== TABEL PERBANDINGAN KOMPREHENSIF 3 MODEL ===")
print(df_hasil.to_string(index=False))

# Pemilihan model terbaik
model_terpilih_nama = max(hasil_terbaik, key=lambda k: hasil_terbaik[k]["skor_gabungan"])
model_terpilih = hasil_terbaik[model_terpilih_nama]["estimator"]
print(f"\nModel Terpilih untuk Deployment: {model_terpilih_nama} (Skor: {hasil_terbaik[model_terpilih_nama]['skor_gabungan']:.4f})")

## 14.13: Serialisasi Model Menggunakan `joblib`
Menyimpan model terbaik dan seluruh pipeline transformasi ke dalam berkas artefak biner (`model_terbaik.joblib`) agar dapat langsung dimuat oleh aplikasi web produksi tanpa perlu melatih ulang.

In [ ]:
# Praktikum 14.13: Serialisasi Model untuk Deployment
os.makedirs("models", exist_ok=True)
path_model = "models/model_loan_default_best.joblib"
joblib.dump(model_terpilih, path_model)
print(f"Model berhasil diserialisasi dan disimpan ke: {path_model}")

# Verifikasi pemuatan artefak
loaded_model = joblib.load(path_model)
sample = X_test.iloc[:1]
pred_sample = loaded_model.predict(sample)[0]
prob_sample = loaded_model.predict_proba(sample)[0][1]
print(f"Uji Inferensi Artefak: Prediksi={pred_sample}, Probabilitas Default={prob_sample*100:.1f}%")

## 14.14: Aplikasi Web Sederhana Berbasis Streamlit
Berikut adalah kode sumber antarmuka aplikasi web `app.py` yang siap dijalankan menggunakan perintah:
```bash
streamlit run app.py
```

In [ ]:
# Praktikum 14.14: Template Skrip Streamlit (app.py)
streamlit_script = """
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Prediksi Risiko Kredit", layout="centered")
st.title("Sistem Penilaian Risiko Gagal Bayar Kredit")
st.write("Aplikasi Capstone Machine Learning (Bab 14) - Unindra")

model = joblib.load("models/model_loan_default_best.joblib")

pendapatan = st.number_input("Pendapatan Bulanan (Rp)", min_value=3000000, value=8500000, step=500000)
usia = st.slider("Usia Peminjam", min_value=21, max_value=65, value=28)
lama_kerja = st.slider("Lama Bekerja (Tahun)", min_value=0, max_value=35, value=4)
pinjaman = st.number_input("Plafon Pinjaman (Rp)", min_value=5000000, value=30000000, step=5000000)
rasio_cicilan = st.slider("Rasio Beban Cicilan", min_value=0.05, max_value=0.80, value=0.25, step=0.01)
skor_kredit = st.slider("Skor Riwayat Kredit SLIK", min_value=350, max_value=850, value=680)

if st.button("Jalankan Prediksi Risiko"):
    df_in = pd.DataFrame([[pendapatan, usia, lama_kerja, pinjaman, rasio_cicilan, skor_kredit]],
                         columns=["pendapatan_bulanan", "usia", "lama_kerja", "jumlah_pinjaman", "rasio_cicilan", "skor_kredit"])
    pred = model.predict(df_in)[0]
    prob = model.predict_proba(df_in)[0][1]
    
    if pred == 1:
        st.error(f"Risiko Tinggi Gagal Bayar (Probabilitas: {prob:.1%})")
    else:
        st.success(f"Risiko Rendah (Layak Disetujui) (Probabilitas: {prob:.1%})")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_script.strip())
print("Berkas aplikasi web 'app.py' berhasil dibuat dan siap dideploy!")

## 14.15 & 14.16: Interpretasi Hasil & Kesimpulan Proyek
1. **Pentingnya Keseimbangan Metrik**: Akurasi mentah tidak memadai pada kasus gagal bayar karena sifat data yang tidak seimbang (*imbalanced*). Optimalisasi F1-Score dan Recall memastikan bank meminimalkan kasus *False Negative* (nasabah macet yang keliru disetujui).
2. **Kesesuaian Operasional**: Meskipun model berbasis Boosting memiliki skor sedikit lebih tinggi pada dataset tertentu, Random Forest terpilih sebagai model akhir berkat stabilitas tinggi dan skor gabungan multi-kriteria terbaik.
3. **Kesiapan Produksi**: Pipeline yang telah diserialisasi ke `.joblib` menjamin tidak adanya *data leakage* antara data latih dan input inferensi baru pada aplikasi Streamlit maupun portal simulasi WebAssembly di Vercel.